# Prediction-error surrogate и adaptive requery

**Frozen screening + confirmatory experiment · 19 August 2026**

Отдельный notebook новой линии uncertainty-aware planning. Все числа ниже
читаются из сохранённых campaign artifacts; exploratory screening и независимый
confirmatory split не смешиваются.

## 1. Гипотеза

Стандартный planner выбирает один из четырёх stochastic candidates:

\[
i_V=\arg\max_i V_i.
\]

Ранее adaptive `requery_l1_h8` улучшил pooled success, но вредил отдельным
задачам. Новая гипотеза: сокращать open-loop chunk и включать risk-aware
ranking только тогда, когда **до исполнения** ожидается большая ошибка
предсказанного future proprio.

Frozen causal surrogate использует четыре online-признака:

\[
\log \widehat e_q=-2.89194
+0.12337z(\log U^a_q)
+0.13764z(\log U^p_q)
+0.40796z(\log \bar V_q)
+0.30282z(\log D^p_q).
\]

- $U^a_q$: disagreement повторных latent action copies;
- $U^p_q$: disagreement latent future-proprio copies;
- $\bar V_q$: среднее value четырёх candidates;
- $D^p_q$: across-candidate spread predicted future proprio.

Порог $G_q=\mathbb{1}[\widehat e_q\ge\tau_e]$ фиксируется по train quantile.
При alarm исполняется 8 действий вместо 16. Risk-aware candidate вычисляется
как

\[
i_R=\arg\max_i\left[z(V_i)-\lambda z(U^{a,first}_i)\right].
\]

Фактическая next-chunk prediction error используется только для последующего
анализа и не входит в выбор того же chunk.

## 2. Протокол

1. Surrogate обучен на 1430 `max(value)` queries и проверен на 1400 queries из
   непересекающихся denoise-10 episodes: Spearman 0.745, case-controlled
   Spearman 0.649, top-quartile error AUROC 0.771.
2. Screening: 6 LIBERO-PRO boundary cases, 8 paired seeds, 102 configurations,
   816 episode executions. Проверяются три frozen thresholds, две phase fractions
   и пять online gating variants.
3. Выбирается ровно один `surrogate_adaptive` и один
   `non_surrogate_adaptive` метод по

\[
J=\Delta_{pool}+0.5\min_c\Delta_c
-0.02\max(0,Q_{ratio}-1).
\]

4. Confirmatory: замороженные методы, 20 новых paired seeds на 12 cases.
   Шесть cases повторяют boundary-задачи, ещё шесть заранее фиксируют новые
   LIBERO-PRO object/language/swap/task shifts.
5. Primary endpoint: paired success delta к `max(value)`, stratified bootstrap
   CI, exact McNemar и Holm correction. Compute overhead и safety signals
   считаются secondary outcomes.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'experiments':
    PROJECT_ROOT = PROJECT_ROOT.parent

SCREEN = PROJECT_ROOT / 'experiments/campaigns/surrogate_screening_20260819/analysis/adaptive_summary'
CONFIRM = PROJECT_ROOT / 'experiments/campaigns/surrogate_confirmatory_20260819/analysis/adaptive_summary'
MEDIA = PROJECT_ROOT / 'experiments/final_results_media/surrogate_confirmatory_20260819'

selected = pd.read_csv(SCREEN / 'selected_for_confirmatory.csv')
frozen = pd.read_csv(CONFIRM / 'frozen_confirmatory_results.csv')
display(selected[['selection_category', 'strategy_id', 'selection_utility']])
display(frozen[['selection_category', 'strategy_id', 'strategy_success_rate',
                'delta_success_rate', 'delta_ci_low', 'delta_ci_high',
                'mcnemar_holm_p', 'query_overhead_ratio']])

## 3. Screening selection

| Category               | Selected strategy                           | max(value)   | strategy   | delta    | worst case   | query cost   | requery rate   | surrogate alarm   |   Utility J |
|:-----------------------|:--------------------------------------------|:-------------|:-----------|:---------|:-------------|:-------------|:---------------|:------------------|------------:|
| non_surrogate_adaptive | requery_l1_h8                               | 32/48        | 39/48      | +14.6 pp | +0.0 pp      | 1.25x        | 39.4%          | 13.9%             |   0.140809  |
| surrogate_adaptive     | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 32/48        | 36/48      | +8.3 pp  | +0.0 pp      | 1.12x        | 20.1%          | 20.1%             |   0.0808356 |

![Screening success/compute trade-off](campaigns/surrogate_screening_20260819/analysis/adaptive_summary/plots/success_compute_tradeoff.png)

Это calibration-результат, а не финальная оценка эффекта.

## 4. Frozen confirmatory result

| Category               | Strategy                                    | max(value)   | strategy   | delta   | 95% CI              | W/L/T     |   Holm p | query cost   | requery rate   | surrogate alarm   |
|:-----------------------|:--------------------------------------------|:-------------|:-----------|:--------|:--------------------|:----------|---------:|:-------------|:---------------|:------------------|
| non_surrogate_adaptive | requery_l1_h8                               | 146/240      | 161/240    | +6.2 pp | [+1.7 pp; +11.2 pp] | 27/12/201 |  0.04741 | 1.27x        | 41.1%          | 14.7%             |
| surrogate_adaptive     | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 146/240      | 147/240    | +0.4 pp | [-4.2 pp; +5.0 pp]  | 17/16/207 |  1       | 1.11x        | 17.7%          | 17.7%             |

![Confirmatory pooled delta](campaigns/surrogate_confirmatory_20260819/analysis/adaptive_summary/plots/pooled_strategy_delta.png)

## 5. Перенос по задачам

### Paired delta по каждому case

| Case                                                 | Method                                      | baseline   | strategy   | delta    | queries   |
|:-----------------------------------------------------|:--------------------------------------------|:-----------|:-----------|:---------|:----------|
| goal_mug_task9_init0_surrogate_confirm               | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 20/20      | 18/20      | -10.0 pp | 1.07x     |
| goal_mug_task9_init0_surrogate_confirm               | requery_l1_h8                               | 20/20      | 17/20      | -15.0 pp | 1.24x     |
| long_milk_task9_init0_surrogate_confirm              | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 16/20      | 18/20      | +10.0 pp | 1.06x     |
| long_milk_task9_init0_surrogate_confirm              | requery_l1_h8                               | 16/20      | 17/20      | +5.0 pp  | 1.29x     |
| long_mug_task4_init0_surrogate_confirm               | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 17/20      | 18/20      | +5.0 pp  | 1.01x     |
| long_mug_task4_init0_surrogate_confirm               | requery_l1_h8                               | 17/20      | 20/20      | +15.0 pp | 1.23x     |
| milk_task5_init0_surrogate_confirm                   | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 9/20       | 5/20       | -20.0 pp | 1.21x     |
| milk_task5_init0_surrogate_confirm                   | requery_l1_h8                               | 9/20       | 9/20       | +0.0 pp  | 1.24x     |
| new_ood_goal_task_task6_init0_surrogate_confirm      | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 0/20       | 0/20       | +0.0 pp  | 1.03x     |
| new_ood_goal_task_task6_init0_surrogate_confirm      | requery_l1_h8                               | 0/20       | 0/20       | +0.0 pp  | 1.34x     |
| new_ood_long_swap_task4_init0_surrogate_confirm      | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 0/20       | 0/20       | +0.0 pp  | 1.20x     |
| new_ood_long_swap_task4_init0_surrogate_confirm      | requery_l1_h8                               | 0/20       | 0/20       | +0.0 pp  | 1.30x     |
| new_ood_object_object_task7_init0_surrogate_confirm  | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 20/20      | 20/20      | +0.0 pp  | 1.00x     |
| new_ood_object_object_task7_init0_surrogate_confirm  | requery_l1_h8                               | 20/20      | 20/20      | +0.0 pp  | 1.23x     |
| new_ood_spatial_lan_task6_init0_surrogate_confirm    | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 20/20      | 20/20      | +0.0 pp  | 1.00x     |
| new_ood_spatial_lan_task6_init0_surrogate_confirm    | requery_l1_h8                               | 20/20      | 20/20      | +0.0 pp  | 1.19x     |
| new_ood_spatial_object_task0_init0_surrogate_confirm | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 20/20      | 20/20      | +0.0 pp  | 1.04x     |
| new_ood_spatial_object_task0_init0_surrogate_confirm | requery_l1_h8                               | 20/20      | 20/20      | +0.0 pp  | 1.27x     |
| new_ood_spatial_swap_task8_init0_surrogate_confirm   | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 1/20       | 1/20       | +0.0 pp  | 1.27x     |
| new_ood_spatial_swap_task8_init0_surrogate_confirm   | requery_l1_h8                               | 1/20       | 7/20       | +30.0 pp | 1.36x     |
| spatial_mug_task0_init0_surrogate_confirm            | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 15/20      | 18/20      | +15.0 pp | 1.16x     |
| spatial_mug_task0_init0_surrogate_confirm            | requery_l1_h8                               | 15/20      | 20/20      | +25.0 pp | 1.25x     |
| yellow_task8_init0_surrogate_confirm                 | phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 8/20       | 9/20       | +5.0 pp  | 1.24x     |
| yellow_task8_init0_surrogate_confirm                 | requery_l1_h8                               | 8/20       | 11/20      | +15.0 pp | 1.26x     |

![Per-case heatmap](campaigns/surrogate_confirmatory_20260819/analysis/adaptive_summary/plots/paired_delta_heatmap.png)

### Повторённые boundary cases и новые OOD axes

| Stratum         | Method                                      |   Paired seeds | delta   | 95% CI              |
|:----------------|:--------------------------------------------|---------------:|:--------|:--------------------|
| known_boundary  | requery_l1_h8                               |            120 | +7.5 pp | [-1.7 pp; +16.7 pp] |
| known_boundary  | phase_surrogate_l1_r0.5_e0.0884176756291_h8 |            120 | +0.8 pp | [-7.5 pp; +10.0 pp] |
| new_ood_holdout | requery_l1_h8                               |            120 | +5.0 pp | [+1.7 pp; +8.3 pp]  |
| new_ood_holdout | phase_surrogate_l1_r0.5_e0.0884176756291_h8 |            120 | +0.0 pp | [-2.5 pp; +2.5 pp]  |

У `max(value)` смешанные success/fail наблюдаются в
**6/12** cases. Cases с 0% или 100%
success полезны для проверки transfer и catastrophic regression, но дают мало
информации о тонком ранжировании planners.

## 6. Mechanism diagnostics

Эти результаты exploratory и не меняют frozen planner.

### Frozen surrogate transfer

| case_stratum    |   queries |   cases |   raw_spearman |   case_controlled_rank_correlation |   case_relative_top_quartile_auc |   alarm_rate |   alarm_precision_top_quartile |   alarm_recall_top_quartile |   median_predicted_error |   median_actual_error |   median_prediction_to_actual_ratio |   median_actual_error_alarm |   median_actual_error_no_alarm |   alarm_actual_error_lift |   mean_absolute_log_error |
|:----------------|----------:|--------:|---------------:|-----------------------------------:|---------------------------------:|-------------:|-------------------------------:|----------------------------:|-------------------------:|----------------------:|------------------------------------:|----------------------------:|-------------------------------:|--------------------------:|--------------------------:|
| all             |      3332 |      12 |          0.644 |                              0.579 |                            0.731 |        0.161 |                          0.540 |                       0.347 |                    0.052 |                 0.047 |                               1.116 |                       0.120 |                          0.040 |                     3.033 |                     0.667 |
| known_boundary  |      1595 |       6 |          0.680 |                              0.638 |                            0.753 |        0.186 |                          0.571 |                       0.421 |                    0.047 |                 0.044 |                               1.083 |                       0.133 |                          0.034 |                     3.872 |                     0.696 |
| new_ood_holdout |      1737 |       6 |          0.600 |                              0.525 |                            0.705 |        0.139 |                          0.502 |                       0.278 |                    0.055 |                 0.050 |                               1.099 |                       0.107 |                          0.043 |                     2.474 |                     0.641 |

![Prediction-error correlations](campaigns/surrogate_confirmatory_20260819/analysis/adaptive_summary/plots/uncertainty_prediction_error_correlations.png)

| online_metric                                          | prediction_error                   |   queries |   cases |   raw_spearman |   case_controlled_rank_correlation |
|:-------------------------------------------------------|:-----------------------------------|----------:|--------:|---------------:|-----------------------------------:|
| planning_predicted_proprio_error                       | prediction_error_future_proprio_l2 |      3332 |      12 |       0.644137 |                           0.579195 |
| candidate_value_mean                                   | prediction_error_future_proprio_l2 |      3332 |      12 |       0.438593 |                           0.471594 |
| latent_action_copy_std_mean_mean_over_samples          | prediction_error_future_proprio_l2 |      3332 |      12 |       0.482544 |                           0.412716 |
| candidate_value_mean                                   | prediction_error_future_wrist_mse  |      3332 |      12 |       0.235346 |                           0.398765 |
| planning_predicted_proprio_error                       | prediction_error_future_wrist_mse  |      3332 |      12 |       0.330661 |                           0.341351 |
| latent_value_element_std_mean_mean_over_samples        | prediction_error_future_proprio_l2 |      3332 |      12 |       0.419237 |                           0.327977 |
| latent_action_first_step_copy_l2_std_mean_over_samples | prediction_error_future_proprio_l2 |      3332 |      12 |       0.435852 |                           0.326662 |
| candidate_action_internal_consistency_mean             | prediction_error_future_proprio_l2 |      3332 |      12 |       0.435852 |                           0.326662 |
| latent_value_element_std_mean_mean_over_samples        | prediction_error_future_wrist_mse  |      3332 |      12 |       0.128127 |                           0.22693  |
| candidate_action_consensus_first_mean                  | prediction_error_future_proprio_l2 |      3332 |      12 |       0.290652 |                           0.199618 |
| action_first_step_l2_std                               | prediction_error_future_proprio_l2 |      3332 |      12 |       0.281379 |                           0.191551 |
| candidate_value_mean                                   | prediction_error_future_image_mse  |      3332 |      12 |      -0.175201 |                           0.166477 |

![Early failure AUROC](campaigns/surrogate_confirmatory_20260819/analysis/adaptive_summary/plots/early_failure_predictor_auc.png)

| feature                                                      |   episodes |   failures |   cases |   raw_auc_high_predicts_fail |   case_controlled_auc_high_predicts_fail |   case_controlled_oriented_auc | risk_direction   |
|:-------------------------------------------------------------|-----------:|-----------:|--------:|-----------------------------:|-----------------------------------------:|-------------------------------:|:-----------------|
| latent_action_first_step_copy_l2_std_mean_over_samples__mean |        240 |         94 |      12 |                     0.308073 |                                 0.546634 |                       0.546634 | high             |
| candidate_action_internal_consistency_mean__mean             |        240 |         94 |      12 |                     0.308073 |                                 0.546634 |                       0.546634 | high             |
| candidate_action_internal_consistency_mean__max              |        240 |         94 |      12 |                     0.309458 |                                 0.544666 |                       0.544666 | high             |
| latent_action_first_step_copy_l2_std_mean_over_samples__max  |        240 |         94 |      12 |                     0.309458 |                                 0.544666 |                       0.544666 | high             |
| candidate_value_mean__mean                                   |        240 |         94 |      12 |                     0.534392 |                                 0.542772 |                       0.542772 | high             |
| planning_predicted_proprio_error__delta                      |        240 |         94 |      12 |                     0.593486 |                                 0.457228 |                       0.542772 | low              |
| latent_value_element_std_mean_mean_over_samples__mean        |        240 |         94 |      12 |                     0.42371  |                                 0.463422 |                       0.536578 | low              |
| value_range__mean                                            |        240 |         94 |      12 |                     0.475372 |                                 0.464806 |                       0.535194 | low              |
| latent_value_element_std_mean_mean_over_samples__max         |        240 |         94 |      12 |                     0.490528 |                                 0.465462 |                       0.534538 | low              |
| value_std__mean                                              |        240 |         94 |      12 |                     0.475153 |                                 0.465753 |                       0.534247 | low              |
| candidate_action_consensus_first_mean__delta                 |        240 |         94 |      12 |                     0.4181   |                                 0.534028 |                       0.534028 | high             |
| action_first_step_l2_std__max                                |        240 |         94 |      12 |                     0.45016  |                                 0.473914 |                       0.526086 | low              |

## 7. Task-failure and interaction diagnostics

| Method                                      | Success   | Failure labels                                                                                                                                   |   Target-drop flags |   Wrong-object flags |   Mean queries |
|:--------------------------------------------|:----------|:-------------------------------------------------------------------------------------------------------------------------------------------------|--------------------:|---------------------:|---------------:|
| action_l1                                   | 151/240   | target_drop_candidate: 51; wrong_object_interaction_candidate: 20; timeout_no_goal: 14; kinematic_deadlock_candidate: 2; timeout_partial_goal: 2 |                  75 |                   37 |          13.73 |
| max_value                                   | 146/240   | target_drop_candidate: 54; wrong_object_interaction_candidate: 22; timeout_no_goal: 14; kinematic_deadlock_candidate: 2; timeout_partial_goal: 2 |                  73 |                   36 |          13.88 |
| phase_surrogate_l1_r0.5_e0.0884176756291_h8 | 147/240   | timeout_no_goal: 37; target_drop_candidate: 33; wrong_object_interaction_candidate: 20; kinematic_deadlock_candidate: 2; timeout_partial_goal: 1 |                  57 |                   35 |          15.65 |
| requery_l1_h8                               | 161/240   | target_drop_candidate: 30; timeout_no_goal: 24; wrong_object_interaction_candidate: 21; timeout_partial_goal: 3; kinematic_deadlock_candidate: 1 |                  51 |                   39 |          16.90 |

### Paired failure-mode changes

| Method                                      | Event                              | baseline rate   | strategy rate   | delta   | reduced/increased   |   exact p |
|:--------------------------------------------|:-----------------------------------|:----------------|:----------------|:--------|:--------------------|----------:|
| action_l1                                   | target_drop_candidate              | 30.4%           | 31.2%           | +0.8 pp | 10/12               | 0.8318    |
| action_l1                                   | wrong_object_interaction_candidate | 15.0%           | 15.4%           | +0.4 pp | 4/5                 | 1         |
| action_l1                                   | timeout_no_goal                    | 5.8%            | 5.8%            | +0.0 pp | 6/6                 | 1         |
| phase_surrogate_l1_r0.5_e0.0884176756291_h8 | target_drop_candidate              | 30.4%           | 23.8%           | -6.7 pp | 28/12               | 0.01659   |
| phase_surrogate_l1_r0.5_e0.0884176756291_h8 | wrong_object_interaction_candidate | 15.0%           | 14.6%           | -0.4 pp | 6/5                 | 1         |
| phase_surrogate_l1_r0.5_e0.0884176756291_h8 | timeout_no_goal                    | 5.8%            | 15.4%           | +9.6 pp | 3/26                | 1.524e-05 |
| requery_l1_h8                               | target_drop_candidate              | 30.4%           | 21.2%           | -9.2 pp | 28/6                | 0.0001951 |
| requery_l1_h8                               | wrong_object_interaction_candidate | 15.0%           | 16.2%           | +1.2 pp | 4/7                 | 0.5488    |
| requery_l1_h8                               | timeout_no_goal                    | 5.8%            | 10.0%           | +4.2 pp | 10/20               | 0.09874   |

`Target-drop` и `wrong-object` здесь являются эвристическими признаками из
LIBERO-PRO rollout. Это secondary diagnostics, а не официальные ограничения
LIBERO-Safety и не самостоятельное доказательство причины task failure.
`exact p` здесь не скорректирован за множественные exploratory comparisons.

## 8. Matched-seed video replays

Discordant outcomes найдены, и механически выбраны **7** matched-seed replay groups. MP4 пока не сгенерированы: replay campaign ожидает свободную GPU 2-7. Confirmatory success statistics уже завершена и от replay не зависит.

## 9. Выводы

- **`requery_l1_h8`:** 161/240 против 146/240, +6.2 pp, 95% CI [+1.7 pp; +11.2 pp], Holm p=0.04741, query cost 1.27x; строго подтверждено.
- **`phase_surrogate_l1_r0.5_e0.0884176756291_h8`:** 147/240 против 146/240, +0.4 pp, 95% CI [-4.2 pp; +5.0 pp], Holm p=1, query cost 1.11x; направление оценено, но строгого подтверждения нет.
- **Control `action_l1`:** +2.1 pp, CI [-2.9 pp; +7.1 pp], query cost 1.00x.
- **Информативность cases:** mixed success/fail есть в 6/12 baseline cases; остальные находятся на 0% или 100% success и проверяют перенос, но почти не различают planners.
- **Surrogate-gated planning hypothesis:** не подтверждена на frozen split.
- **Surrogate failure-mode diagnostic:** heuristic target-drop flags 73 -> 57, while `timeout_no_goal` failures 14 -> 37. Это возможный обмен drop-risk на незавершение задачи, а не официальный LIBERO-Safety результат.
- **Confirmed requery failure modes:** heuristic target-drop flags 73 -> 51, while `timeout_no_goal` failures 14 -> 24; requery уменьшает drops, но частично переносит ошибки в незавершение.
- **Mechanism:** surrogate переносится на next-chunk future-proprio error (case-controlled rho=0.579, AUROC=0.731, alarm lift=3.03x), но эта физическая ошибка не тождественна task failure.
- **Early task-fail detection:** лучший q0-q3 case-controlled AUROC равен 0.547; универсальный online fail detector по текущим метрикам не получен.

Интерпретация ограничивается frozen confirmatory split. Положительный screening
delta без переноса на новые seeds/cases не считается доказанным улучшением.
Prediction-error surrogate оценивает риск динамически на каждом query, но сам по
себе не гарантирует task failure: он должен рассматриваться как routing signal
для candidate ranking и частоты обратной связи со средой.

## 10. Воспроизводимость

- [Frozen hypotheses and protocol](SURROGATE_REQUERY_HYPOTHESES_20260819.md)
- [Surrogate artifact](models/future_proprio_error_surrogate_v1.json)
- [Screening analysis](campaigns/surrogate_screening_20260819/analysis/adaptive_summary/README.md)
- [Confirmatory analysis](campaigns/surrogate_confirmatory_20260819/analysis/adaptive_summary/README.md)
- [Queued video selection](configs/libero_campaign_surrogate_video_replays.csv)
- [Video manifest](final_results_media/surrogate_confirmatory_20260819/README.md)